In [1]:
# 04_baf_baseline — Cell 1: load clean BAF, chronological split (months 0-4 / 5 / 6-7), save split summary
import sys, json; sys.path.insert(0, "..")
import pandas as pd
import numpy as np
from src.config import PATHS, SEED, Timer

with Timer("read BAF parquet"):
    baf = pd.read_parquet(PATHS["work"] / "baf_base_clean.parquet")

TARGET = "fraud_bool"
parts = {"train": baf[baf["month"] <= 4],
         "valid": baf[baf["month"] == 5],
         "test":  baf[baf["month"] >= 6]}
summary_split = {k: {"months": sorted(map(int, d["month"].unique())), "n": int(len(d)),
                     "n_fraud": int(d[TARGET].sum()), "prevalence": round(float(d[TARGET].mean()), 5)}
                 for k, d in parts.items()}
for k, v in summary_split.items():
    print(k, v)

assert sum(len(d) for d in parts.values()) == len(baf)
assert parts["train"]["month"].max() < parts["valid"]["month"].min() < parts["test"]["month"].min()
(PATHS["repo"] / "reports" / "split_baf.json").write_text(json.dumps(summary_split, indent=2))
print("\nsplit OK: no month overlap, no row lost; saved reports/split_baf.json")

[read BAF parquet] wall = 3.5 s | RSS now = 0.61 GB | peak RSS = 0.61 GB
train {'months': [0, 1, 2, 3, 4], 'n': 675666, 'n_fraud': 6740, 'prevalence': 0.00998}
valid {'months': [5], 'n': 119323, 'n_fraud': 1411, 'prevalence': 0.01183}
test {'months': [6, 7], 'n': 205011, 'n_fraud': 2878, 'prevalence': 0.01404}

split OK: no month overlap, no row lost; saved reports/split_baf.json


In [2]:
%%writefile ../src/metrics.py
"""Ranking metrics shared by all notebooks (BAF and AML). Higher score = more suspicious."""
import numpy as np
from sklearn.metrics import average_precision_score, roc_curve


def pr_auc(y, s):
    return float(average_precision_score(y, s))


def recall_at_fpr(y, s, fpr_target=0.05):
    """Best recall (TPR) reachable while FPR <= fpr_target.
    With few negatives the FPR grid is coarse (step = 1/n_neg); meaningful on full splits."""
    fpr, tpr, _ = roc_curve(y, s)
    return float(tpr[fpr <= fpr_target].max())


def threshold_at_fpr(y, s, fpr_target=0.05):
    """Score cut-off that flags fpr_target of the negatives.
    Use on VALIDATION to pick an operating threshold; apply unchanged on test."""
    return float(np.quantile(np.asarray(s)[np.asarray(y) == 0], 1 - fpr_target))


def precision_at_k(y, s, k=100):
    top = np.argsort(-np.asarray(s), kind="stable")[:k]
    return float(np.asarray(y)[top].mean())


def lift_at_frac(y, s, frac=0.01):
    k = max(1, int(len(y) * frac))
    return precision_at_k(y, s, k) / float(np.mean(y))


def summary(y, s, ks=(100, 500)):
    out = {"prevalence": float(np.mean(y)), "pr_auc": pr_auc(y, s),
           "recall_at_5fpr": recall_at_fpr(y, s, 0.05), "lift_at_1pct": lift_at_frac(y, s, 0.01)}
    for k in ks:
        out[f"precision_at_{k}"] = precision_at_k(y, s, k)
    return out

Writing ../src/metrics.py


In [3]:
# 04_baf_baseline — Cell 3: test the metric functions against hand-computed values
import importlib, src.metrics as M
importlib.reload(M)

y = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 1])
s = np.array([.1, .2, .3, .4, .5, .6, .7, .9, .8, .95])   # one negative (0.9) outranks a positive
got = M.summary(y, s, ks=(1, 3))
got["threshold_at_fpr_0.125"] = M.threshold_at_fpr(y, s, 0.125)
expected = {"prevalence": 0.2, "pr_auc": 0.8333, "recall_at_5fpr": 0.5, "lift_at_1pct": 5.0,
            "precision_at_1": 1.0, "precision_at_3": 0.6667, "threshold_at_fpr_0.125": 0.725}
for k, e in expected.items():
    ok = abs(got[k] - e) < 1e-3
    print(f"{k:24s} got={got[k]:.4f}  expected={e:.4f}  {'OK' if ok else 'MISMATCH'}")
    assert ok, k
print("\nALL METRIC TESTS PASSED")

prevalence               got=0.2000  expected=0.2000  OK
pr_auc                   got=0.8333  expected=0.8333  OK
recall_at_5fpr           got=0.5000  expected=0.5000  OK
lift_at_1pct             got=5.0000  expected=5.0000  OK
precision_at_1           got=1.0000  expected=1.0000  OK
precision_at_3           got=0.6667  expected=0.6667  OK
threshold_at_fpr_0.125   got=0.7250  expected=0.7250  OK

ALL METRIC TESTS PASSED


In [5]:
# 04_baf_baseline — Cell 4: feature matrices for the three representations of intended_balcon_amount
MINUS1_COLS = ["prev_address_months_count", "current_address_months_count", "bank_months_count",
               "session_length_in_minutes", "device_distinct_emails_8w"]
BALCON = "intended_balcon_amount"
CAT_COLS = baf.select_dtypes("category").columns.tolist()
RAW = [c for c in baf.columns if c not in (TARGET, "month")]
assert len(RAW) == 29, len(RAW)

def make_X(d, variant):
    """Build features for variant A, B or C (DECISION #5 / #8). Uses only the row itself."""
    X = d[RAW].copy()
    for c in MINUS1_COLS:
        X[f"{c}_is_missing"] = (d[c] < 0).astype("int8")
    if variant in ("B", "C"):
        X[f"{BALCON}_is_missing"] = (d[BALCON] < 0).astype("int8")
    if variant == "C":
        X.loc[d[BALCON] < 0, BALCON] = np.nan
    return X

y = {k: d[TARGET].to_numpy() for k, d in parts.items()}
for v in "ABC":
    X = make_X(parts["train"], v)
    print(f"{v}: shape={X.shape}  balcon NaN={int(X[BALCON].isna().sum()):,}  "
          f"flags={[c for c in X.columns if c.endswith('_is_missing')].__len__()}")
print("categorical:", CAT_COLS)

A: shape=(675666, 34)  balcon NaN=0  flags=5
B: shape=(675666, 35)  balcon NaN=0  flags=6
C: shape=(675666, 35)  balcon NaN=493,751  flags=6
categorical: ['payment_type', 'employment_status', 'housing_status', 'source', 'device_os']


In [6]:
# 04_baf_baseline — Cell 5: train A/B/C with DECISION #8 settings; compare on validation (month 5) only
import lightgbm as lgb

PARAMS = dict(objective="binary", learning_rate=0.05, num_leaves=63, min_child_samples=100,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              metric="average_precision", seed=SEED, deterministic=True,
              force_col_wise=True, num_threads=8, verbose=-1)

def fit(variant):
    Xtr, Xva = make_X(parts["train"], variant), make_X(parts["valid"], variant)
    dtr = lgb.Dataset(Xtr, y["train"], categorical_feature=CAT_COLS, free_raw_data=False)
    dva = lgb.Dataset(Xva, y["valid"], categorical_feature=CAT_COLS, reference=dtr)
    booster = lgb.train(PARAMS, dtr, num_boost_round=2000, valid_sets=[dva],
                        callbacks=[lgb.early_stopping(100, verbose=False)])
    return booster, booster.predict(Xva, num_iteration=booster.best_iteration)

models, rows = {}, []
for v in "ABC":
    with Timer(f"train variant {v}"):
        booster, s_va = fit(v)
    models[v] = booster
    rows.append({"variant": v, "best_iter": booster.best_iteration,
                 "lgb_ap": booster.best_score["valid_0"]["average_precision"],
                 **M.summary(y["valid"], s_va)})
val = pd.DataFrame(rows).set_index("variant")
print(val.round(4).to_string())

# Pre-registered rule (DECISION #5): best validation PR-AUC; within 5% of best -> simplest (A < B < C)
best = val["pr_auc"].max()
within = [v for v in "ABC" if best - val.loc[v, "pr_auc"] < 0.05 * best]
WINNER = within[0]
print(f"\nbest PR-AUC = {best:.4f} | within 5%: {within} | WINNER = {WINNER}")
print("max |lgb_ap - sklearn pr_auc| =", float((val["lgb_ap"] - val["pr_auc"]).abs().max()))

[train variant A] wall = 10.3 s | RSS now = 1.03 GB | peak RSS = 1.45 GB
[train variant B] wall = 9.1 s | RSS now = 1.03 GB | peak RSS = 1.47 GB
[train variant C] wall = 12.9 s | RSS now = 1.04 GB | peak RSS = 1.47 GB
         best_iter  lgb_ap  prevalence  pr_auc  recall_at_5fpr  lift_at_1pct  precision_at_100  precision_at_500
variant                                                                                                         
A              141  0.1852      0.0118  0.1852          0.5393       24.8808              0.56             0.360
B              115  0.1831      0.0118  0.1831          0.5393       22.8960              0.57             0.362
C              164  0.1878      0.0118  0.1878          0.5330       23.6757              0.59             0.360

best PR-AUC = 0.1878 | within 5%: ['A', 'B', 'C'] | WINNER = A
max |lgb_ap - sklearn pr_auc| = 3.0531133177191805e-16


In [7]:
# 04_baf_baseline — Cell 6: FINAL — evaluate the winner ONCE on test (months 6-7); fairness report
import json
final = models[WINNER]
s_va = final.predict(make_X(parts["valid"], WINNER), num_iteration=final.best_iteration)
s_te = final.predict(make_X(parts["test"], WINNER), num_iteration=final.best_iteration)

thr = M.threshold_at_fpr(y["valid"], s_va, 0.05)          # operating point chosen on validation
res = M.summary(y["test"], s_te)
flag, neg = s_te >= thr, y["test"] == 0
old = parts["test"]["customer_age"].to_numpy() >= 50
fpr_old, fpr_young = float(flag[neg & old].mean()), float(flag[neg & ~old].mean())
res.update(pr_auc_x_prevalence=res["pr_auc"] / res["prevalence"],
           valid_threshold=thr, fpr_at_valid_thr=float(flag[neg].mean()),
           recall_at_valid_thr=float(flag[~neg].mean()),
           fpr_age_ge50=fpr_old, fpr_age_lt50=fpr_young,
           fpr_ratio_min_over_max=min(fpr_old, fpr_young) / max(fpr_old, fpr_young),
           n_neg_age_ge50=int((neg & old).sum()), n_fraud_age_ge50=int((~neg & old).sum()))
for k, v in res.items():
    print(f"{k:24s} {v:.4f}" if isinstance(v, float) else f"{k:24s} {v}")

# Pre-registered BAF gate (DECISION #2 / #2a); leakage review follows in the next cell
print("\nGATE  Recall@5%FPR >= 0.40 :", res["recall_at_5fpr"] >= 0.40)
print("GATE  PR-AUC >= 0.070      :", res["pr_auc"] >= 0.070)

# Persist everything needed later (SHAP, calibration, report) — model and scores outside git
final.save_model(str(PATHS["work"] / f"baf_lgbm_{WINNER}.txt"), num_iteration=final.best_iteration)
pd.DataFrame({"month": parts["test"]["month"].to_numpy(), "y": y["test"], "score": s_te}).to_parquet(
    PATHS["work"] / "baf_test_scores.parquet", index=False)
out = {"winner": WINNER, "best_iteration": final.best_iteration,
       "validation_abc": val.reset_index().to_dict(orient="records"), "test": res}
(PATHS["repo"] / "reports" / "results_baf_baseline.json").write_text(json.dumps(out, indent=2))
print("\nsaved: model + test scores (work/), results_baf_baseline.json (reports/)")

prevalence               0.0140
pr_auc                   0.1829
recall_at_5fpr           0.5452
lift_at_1pct             20.6752
precision_at_100         0.5400
precision_at_500         0.4120
pr_auc_x_prevalence      13.0259
valid_threshold          0.0280
fpr_at_valid_thr         0.0603
recall_at_valid_thr      0.5754
fpr_age_ge50             0.1400
fpr_age_lt50             0.0449
fpr_ratio_min_over_max   0.3209
n_neg_age_ge50           32790
n_fraud_age_ge50         989

GATE  Recall@5%FPR >= 0.40 : True
GATE  PR-AUC >= 0.070      : True

saved: model + test scores (work/), results_baf_baseline.json (reports/)


In [8]:
# 04_baf_baseline — Cell 7: leakage checks for the BAF baseline (train/valid only; test not used)
assert "month" not in final.feature_name() and TARGET not in final.feature_name()

# 1) Feature importance: a single feature with overwhelming gain would be suspicious
imp = pd.Series(final.feature_importance("gain"), index=final.feature_name())
imp = (100 * imp / imp.sum()).sort_values(ascending=False)
print("top 10 features by gain share (%):")
print(imp.head(10).round(2).to_string())

# 2) Label-permutation test: same pipeline with shuffled train labels must score ~ prevalence
rng = np.random.default_rng(SEED)
X_tr, X_va = make_X(parts["train"], WINNER), make_X(parts["valid"], WINNER)
d_perm = lgb.Dataset(X_tr, rng.permutation(y["train"]), categorical_feature=CAT_COLS)
d_va = lgb.Dataset(X_va, y["valid"], categorical_feature=CAT_COLS, reference=d_perm)
with Timer("permuted-label model"):
    perm = lgb.train(PARAMS, d_perm, num_boost_round=2000, valid_sets=[d_va],
                     callbacks=[lgb.early_stopping(100, verbose=False)])
ap_perm = M.pr_auc(y["valid"], perm.predict(X_va, num_iteration=perm.best_iteration))
print(f"\npermuted-label validation PR-AUC = {ap_perm:.4f} | prevalence = {y['valid'].mean():.4f} "
      f"| real model = {val.loc[WINNER, 'pr_auc']:.4f}")
print("LEAKAGE CHECK:", "PASS" if ap_perm < 2 * y["valid"].mean() else "INVESTIGATE")

top 10 features by gain share (%):
housing_status                      16.26
device_os                            7.87
name_email_similarity                6.05
credit_risk_score                    5.40
current_address_months_count         5.37
days_since_request                   3.67
intended_balcon_amount               3.66
bank_branch_count_8w                 3.65
velocity_4w                          3.56
date_of_birth_distinct_emails_4w     3.52
[permuted-label model] wall = 4.4 s | RSS now = 1.02 GB | peak RSS = 1.47 GB

permuted-label validation PR-AUC = 0.0130 | prevalence = 0.0118 | real model = 0.1852
LEAKAGE CHECK: PASS
